In [1]:
import os
import subprocess
from tqdm import tqdm

def extract_meld_audio_ffmpeg(base_path, output_root):
    """
    Extracts mono audio from MELD mp4 files using FFmpeg.
    Standardizes output to 16kHz wav for Wav2Vec2 compatibility.
    """
    sub_folders = ['train', 'dev', 'test']
    
    for folder in sub_folders:
        input_dir = os.path.join(base_path, folder)
        output_dir = os.path.join(output_root, folder)
        
        # Create output sub-directories if they don't exist
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"📁 Created directory: {output_dir}")

        if not os.path.exists(input_dir):
            print(f"⚠️ Skipping {folder}: Source path not found.")
            continue

        print(f"🚀 Extracting audio from {folder} via FFmpeg...")
        video_files = [f for f in os.listdir(input_dir) if f.endswith('.mp4')]
        
        for video_file in tqdm(video_files):
            video_path = os.path.normpath(os.path.join(input_dir, video_file))
            # Convert filename extension from .mp4 to .wav
            audio_file_name = video_file.replace('.mp4', '.wav')
            audio_output_path = os.path.normpath(os.path.join(output_dir, audio_file_name))
            
            # Skip if file already exists (useful for resuming interrupted tasks)
            if os.path.exists(audio_output_path):
                continue

            # FFmpeg Command breakdown:
            # -y: Overwrite output files without asking
            # -i: Input file path
            # -vn: Disable video recording (extract audio only)
            # -ac 1: Set audio channels to 1 (Mono - critical for 5.1 audio sources)
            # -ar 16000: Set audio sampling rate to 16kHz (Standard for Wav2Vec2)
            # -loglevel error: Suppress non-critical logs
            command = [
                'ffmpeg',
                '-y',
                '-i', video_path,
                '-vn',
                '-ac', '1',
                '-ar', '16000',
                audio_output_path,
                '-loglevel', 'error'
            ]

            try:
                # Execute the shell command
                subprocess.run(command, check=True)
            except subprocess.CalledProcessError as e:
                print(f"❌ File corrupted or FFmpeg error at {video_file}: {e}")
            except FileNotFoundError:
                print("🚨 FFmpeg not found! Please ensure it's installed and added to PATH.")
                print("Hint: Run 'conda install ffmpeg' or 'sudo apt install ffmpeg'.")
                return

# Execution
if __name__ == "__main__":
    extract_meld_audio_ffmpeg(
        base_path="./MELD_Vid", 
        output_root="./Raw_Data/MELD"
    )

📁 Created directory: ./Raw_Data/MELD/train
🚀 Extracting audio from train via FFmpeg...


 56%|█████▋    | 5643/9989 [03:45<02:56, 24.62it/s][mov,mp4,m4a,3gp,3g2,mj2 @ 0x5cdc61e6af40] moov atom not found
[in#0 @ 0x5cdc61e6ae40] Error opening input: Invalid data found when processing input
Error opening input file MELD_Vid/train/dia125_utt3.mp4.
Error opening input files: Invalid data found when processing input
 57%|█████▋    | 5649/9989 [03:46<03:03, 23.63it/s]

❌ File corrupted or FFmpeg error at dia125_utt3.mp4: Command '['ffmpeg', '-y', '-i', 'MELD_Vid/train/dia125_utt3.mp4', '-vn', '-ac', '1', '-ar', '16000', 'Raw_Data/MELD/train/dia125_utt3.wav', '-loglevel', 'error']' returned non-zero exit status 183.


100%|██████████| 9989/9989 [06:40<00:00, 24.94it/s]


📁 Created directory: ./Raw_Data/MELD/dev
🚀 Extracting audio from dev via FFmpeg...


100%|██████████| 1112/1112 [00:43<00:00, 25.37it/s]


📁 Created directory: ./Raw_Data/MELD/test
🚀 Extracting audio from test via FFmpeg...


100%|██████████| 2747/2747 [01:50<00:00, 24.92it/s]


In [2]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

class VoxSentinelDataProtocol:
    def __init__(self, root_path="./Raw_Data"):
        self.root_path = root_path
        self.master_list = []
        
        # Unified Emotion Mapping
        self.emotion_map = {
            'joy': 'happy', 'sadness': 'sad', 'anger': 'angry', 
            'fear': 'fear', 'disgust': 'disgust', 'surprise': 'surprise', 'neutral': 'neutral',
            'HAP': 'happy', 'SAD': 'sad', 'ANG': 'angry', 
            'FEA': 'fear', 'DIS': 'disgust', 'NEU': 'neutral',
            'happy': 'happy', 'sad': 'sad', 'angry': 'angry', 'ps': 'surprise',
            'pleasant_surprised': 'surprise'
        }

    def _stratified_split(self, data_list, train_size=0.8, dev_size=0.1):
        """
        Helper to split non-MELD datasets into train/dev/test 
        while maintaining emotion ratios.
        """
        if not data_list:
            return []
            
        df = pd.DataFrame(data_list)
        
        # First split: Train vs (Dev + Test)
        train_df, temp_df = train_test_split(
            df, 
            test_size=(1 - train_size), 
            stratify=df['emotion'], 
            random_state=42
        )
        
        # Second split: Dev vs Test from the temp_df
        # Calculate relative size of dev compared to (dev + test)
        relative_dev_size = dev_size / (1 - train_size)
        dev_df, test_df = train_test_split(
            temp_df, 
            test_size=(1 - relative_dev_size), 
            stratify=temp_df['emotion'], 
            random_state=42
        )
        
        train_df['split'] = 'train'
        dev_df['split'] = 'dev'
        test_df['split'] = 'test'
        
        return pd.concat([train_df, dev_df, test_df]).to_dict('records')

    def process_crema(self):
        print("🔍 Processing CREMA-D...")
        folder = os.path.join(self.root_path, "Crema")
        temp_list = []
        if not os.path.exists(folder): return
        
        for file in os.listdir(folder):
            if file.endswith(".wav"):
                parts = file.split('_')
                if len(parts) > 2:
                    raw_emo = parts[2]
                    if raw_emo in self.emotion_map:
                        temp_list.append({
                            'path': os.path.join(folder, file),
                            'emotion': self.emotion_map[raw_emo],
                            'dataset': 'crema'
                        })
        # Apply 80/10/10 split
        self.master_list.extend(self._stratified_split(temp_list))

    def process_meld(self):
        print("🔍 Processing MELD (Preserving Original Splits)...")
        meld_root = os.path.join(self.root_path, "MELD")
        splits = {'train': 'train_sent_emo.csv', 'dev': 'dev_sent_emo.csv', 'test': 'test_sent_emo.csv'}

        for split_name, csv_filename in splits.items():
            csv_path = os.path.join(meld_root, csv_filename)
            audio_subdir = os.path.join(meld_root, split_name)
            if not os.path.exists(csv_path): continue
            
            df = pd.read_csv(csv_path)
            for _, row in df.iterrows():
                file_name = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.wav"
                audio_path = os.path.normpath(os.path.join(audio_subdir, file_name))
                
                if os.path.exists(audio_path):
                    raw_emo = str(row['Emotion']).lower().strip()
                    if raw_emo in self.emotion_map:
                        self.master_list.append({
                            'path': audio_path,
                            'emotion': self.emotion_map[raw_emo],
                            'dataset': 'meld',
                            'split': split_name
                        })

    def process_ravdess(self):
        print("🔍 Processing RAVDESS...")
        rav_root = os.path.join(self.root_path, "RAVDESS")
        rav_map = {'01':'neutral', '03':'happy', '04':'sad', '05':'angry', '06':'fear', '07':'disgust', '08':'surprise'}
        temp_list = []
        if not os.path.exists(rav_root): return
        
        for root, _, files in os.walk(rav_root):
            for file in files:
                if file.endswith(".wav"):
                    parts = file.split('-')
                    if len(parts) > 2:
                        emo_code = parts[2]
                        if emo_code in rav_map:
                            temp_list.append({
                                'path': os.path.join(root, file),
                                'emotion': rav_map[emo_code],
                                'dataset': 'ravdess'
                            })
        self.master_list.extend(self._stratified_split(temp_list))

    def process_tess(self):
        print("🔍 Processing TESS...")
        tess_root = os.path.join(self.root_path, "Tess")
        temp_list = []
        if not os.path.exists(tess_root): return
        
        for folder in os.listdir(tess_root):
            folder_path = os.path.join(tess_root, folder)
            if os.path.isdir(folder_path):
                raw_emo = folder.split('_')[-1].lower()
                for file in os.listdir(folder_path):
                    if file.endswith(".wav"):
                        if raw_emo in self.emotion_map:
                            temp_list.append({
                                'path': os.path.join(folder_path, file),
                                'emotion': self.emotion_map[raw_emo],
                                'dataset': 'tess'
                            })
        self.master_list.extend(self._stratified_split(temp_list))

    def export(self, output_dir="./Protocol", filename="master_metadata.csv"):
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        # Convert to DataFrame and shuffle EVERYTHING to mix datasets
        final_df = pd.DataFrame(self.master_list)
        final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
        
        output_path = os.path.join(output_dir, filename)
        final_df.to_csv(output_path, index=False)
        
        print("\n" + "="*45)
        print("📊 VOXSENTINEL REBALANCED DATA SUMMARY")
        print("="*45)
        print(f"✅ Total samples: {len(final_df)}")
        print("\n--- Distribution by Split (Balanced) ---")
        print(final_df['split'].value_counts(normalize=True).map(lambda n: f"{n:.1%}"))
        print("\n--- Samples per Split ---")
        print(pd.crosstab(final_df['dataset'], final_df['split']))
        print("\n--- Emotion Distribution across Splits ---")
        print(pd.crosstab(final_df['emotion'], final_df['split']))
        print("="*45)
        print(f"📁 Master CSV saved to: {output_path}")

if __name__ == "__main__":
    protocol = VoxSentinelDataProtocol()
    protocol.process_crema()
    protocol.process_meld() 
    protocol.process_ravdess()
    protocol.process_tess()
    protocol.export()

🔍 Processing CREMA-D...
🔍 Processing MELD (Preserving Original Splits)...
🔍 Processing RAVDESS...
🔍 Processing TESS...

📊 VOXSENTINEL REBALANCED DATA SUMMARY
✅ Total samples: 24996

--- Distribution by Split (Balanced) ---
split
train    76.1%
test     15.0%
dev       8.9%
Name: proportion, dtype: str

--- Samples per Split ---
split     dev  test  train
dataset                   
crema     744   745   5953
meld     1108  2610   9988
ravdess   125   125    998
tess      260   260   2080

--- Emotion Distribution across Splits ---
split     dev  test  train
emotion                   
angry     339   531   2600
disgust   208   254   1762
fear      226   237   1758
happy     350   589   3232
neutral   627  1415   5975
sad       298   394   2173
surprise  189   320   1519
📁 Master CSV saved to: ./Protocol/master_metadata.csv
